# Lesson 03 - Agentic Design Patterns

In this lesson, we explore three foundational design patterns for building effective AI agents:

1. **Selkeät agentin ohjeet** — Tarkkojen, roolia määrittelevien kehotteiden laatiminen, jotka ohjaavat agentin toimintaa  
2. **Rakenneitu tuloste Pydantic-malleilla** — Varmistaminen, että agentit palauttavat ennustettavaa, validoitua dataa  
3. **Yhden vastuun agentit** — Keskittyneiden agenttien suunnittelu, jotka kukin tekevät yhden asian hyvin  

Sovellamme kutakin mallia **matkakohteiden suosittelija** -tilanteeseen, rakentaen vaiheittain järjestelmää, joka voi ehdottaa kohteita, tarkistaa saatavuuden ja hoitaa logistiikan.


## Asetus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Malli 1: Selkeät agentin ohjeet

Vaikutuksiltaan tehokkain malli on myös yksinkertaisin: kirjoittaa selkeät, yksityiskohtaiset ohjeet agentillesi.

Hyvät ohjeet määrittelevät:
- **Kuka** agentti on (persoonallisuus ja sävy)
- **Mitä** sen tulisi tehdä (vastuutehtävät askel askeleelta)
- **Miten** sen tulisi käyttäytyä (rajoitteet ja tyyli)

Alla luomme matkakonsiergipalvelun agentin, jolla on selkeät ohjeet, jotka muovaavat jokaisen sen tuottaman vastauksen.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Malli 2: Rakenteinen tuloste Pydantic-mallien avulla

Vapaa muotoinen teksti on hyödyllistä keskustelussa, mutta jatkokäsittelyjärjestelmät tarvitsevat jäsenneltyä dataa.
Yhdistämällä **Pydantic-mallit** ja **työkalufunktio**, voimme:

- Määritellä tarkan skeeman agentin tulosteelle
- Tarkistaa vastaukset automaattisesti
- Integroimaan agentin tulokset luotettavasti sovelluslogiikkaan

Esittelemme myös työkalun, joka palauttaa kohdetiedot, jotta agentti perustaa suosituksensa todellisiin tietoihin.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Malli 3: Yhden vastuun agentit

Monimutkaiset tehtävät hyötyvät työn jakamisesta useiden keskittyneiden agenttien kesken, joista jokaisella on yksi vastuualue:

- **Kohdetuntemusasiantuntija**, joka tietää paikoista ja saatavuudesta
- **Logistiikkasuunnittelija**, joka hoitaa lennot, hotellit ja matkasuunnitelmat

Tämä heijastaa ohjelmistokehityksen periaatetta *vastuiden erottelusta* — jokainen agentti on helpompi testata, ylläpitää ja parantaa itsenäisesti.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Yhteenveto

Tässä oppitunnissa sovelsimme kolmea agenttipohjaista suunnittelumallia matkasuositusten tilanteessa:

| Malli | Keskeinen ajatus | Hyöty |
|---|---|---|
| **Selkeät ohjeet** | Määritä persoona, vastuut ja rajoitteet etukäteen | Johdonmukainen, brändin mukainen agenttikäytös |
| **Rakenteellinen tuloste** | Käytä Pydantic-malleja vastausmuotona | Varmennettu, koneellisesti luettava tulos |
| **Yksi vastuualue** | Anna jokaiselle agentille yksi keskittynyt tehtävä | Helpompi testata, ylläpitää ja yhdistää |

Nämä mallit toimivat luonnollisesti yhdessä — voit yhdistää selkeät ohjeet ja rakenteellisen tulosteen yksivastuisessa agentissa rakentaaksesi vankkoja, tuotantovalmiita järjestelmiä.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
